# Week 5: Advanced Ensemble Algorithms
## Notebook 13: XGBoost Core Objective & Second-Order Regularization

### 1. Conceptual Intuition & Theoretical Background
While traditional Gradient Boosting relies exclusively on first-order residuals (gradient $g_i$), **XGBoost (Chen & Guestrin, 2016)** uses a **second-order Taylor expansion** of the loss function:
$$\mathcal{L}^{(t)} \approx \sum_{i=1}^{n} \left[ l(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)$$

Where gradients and hessians are:
$$g_i = \frac{\partial l(y_i, \hat{y}_i)}{\partial \hat{y}_i}, \quad h_i = \frac{\partial^2 l(y_i, \hat{y}_i)}{\partial \hat{y}_i^2}$$

**Tree Complexity Penalty ($\Omega$):**
$$\Omega(f) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2$$
- $T$: Number of terminal leaf nodes (penalized by $\gamma$).
- $w_j$: Optimal leaf output weight (penalized by $L_2$ shrinkage parameter $\lambda$):
$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$

**XGBoost Split Gain Formula:**
$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$
A split is only accepted if $\text{Gain} > 0$, providing automated tree pruning.

---

### 2. Scratch Python Implementation: XGBoost Split Evaluator

In [ ]:
import numpy as np

def compute_xgboost_split_gain(g_left, h_left, g_right, h_right, reg_lambda=1.0, gamma=0.5):
    """
    Computes exact XGBoost second-order split gain with L2 and tree complexity regularization.
    """
    G_L = np.sum(g_left)
    H_L = np.sum(h_left)
    G_R = np.sum(g_right)
    H_R = np.sum(h_right)
    
    G_total = G_L + G_R
    H_total = H_L + H_R
    
    # Quality scores: G^2 / (H + lambda)
    score_left = (G_L ** 2) / (H_L + reg_lambda)
    score_right = (G_R ** 2) / (H_R + reg_lambda)
    score_parent = (G_total ** 2) / (H_total + reg_lambda)
    
    gain = 0.5 * (score_left + score_right - score_parent) - gamma
    
    # Optimal leaf weights
    w_L = -G_L / (H_L + reg_lambda)
    w_R = -G_R / (H_R + reg_lambda)
    
    return gain, w_L, w_R

# Demonstration: Evaluating Split Decision
# Simulated Squared Error Loss: g_i = (y_hat - y), h_i = 1.0
np.random.seed(42)
g_candidate_L = np.random.normal(loc=-2.0, scale=0.5, size=20)
h_candidate_L = np.ones(20)

g_candidate_R = np.random.normal(loc=1.8, scale=0.5, size=20)
h_candidate_R = np.ones(20)

# Evaluate split under different regularization settings
gain_low_reg, w_L1, w_R1 = compute_xgboost_split_gain(
    g_candidate_L, h_candidate_L, g_candidate_R, h_candidate_R, reg_lambda=1.0, gamma=0.1
)

gain_high_reg, w_L2, w_R2 = compute_xgboost_split_gain(
    g_candidate_L, h_candidate_L, g_candidate_R, h_candidate_R, reg_lambda=50.0, gamma=5.0
)

print("--- XGBoost Regularized Split Criterion ---")
print(f"Low Regularization (lambda=1, gamma=0.1)  -> Split Gain: {gain_low_reg:.4f} (Split Accepted!)")
print(f"  Leaf Weights: Left={w_L1:.3f}, Right={w_R1:.3f}")
print(f"High Regularization (lambda=50, gamma=5) -> Split Gain: {gain_high_reg:.4f} (Split PRUNED!)")
print(f"  Leaf Weights Shrunk To: Left={w_L2:.3f}, Right={w_R2:.3f}")

### 3. Step-by-Step Code Walkthrough
1. **Hessian Regularization ($\lambda$):** Prevents weight explosion in sparse leaf nodes.
2. **Complexity Pruning ($\gamma$):** Requires split gains to exceed $\gamma$, pruning uninformative branches.
3. **Closed-Form Weights ($w^*$):** Directly computes optimal leaf outputs without step-size line search.

---

### 4. Industry Applications & Trade-offs
- **Kaggle Competitions & High-Throughput Scoring:** XGBoost powers production tabular ranking and risk modeling.
- **Trade-off:** Requires tuning multiple coupled hyperparameters ($\eta, \gamma, \lambda, \alpha$).

---

### 5. Top Interview Questions & Answers
**Q1: Why does XGBoost use second-order gradients (Hessians) while standard gradient boosting uses only first-order gradients?**
> **Answer:** Second-order Taylor expansion provides a quadratic approximation of the loss function, allowing exact analytical calculation of optimal leaf weights and split gain without requiring an extra line-search step.

**Q2: What is the role of parameter `gamma` in XGBoost?**
> **Answer:** `gamma` specifies the minimum loss reduction required to make a further partition on a leaf node. It acts as an automated post-pruning threshold.